<br/>
<img src="images/utfsm.png" alt="" width="130px" align="left"/>
<img src="images/utfsm.png" alt="" width="130px" align="right"/>
<div align="center">
<h1>Arquitecturas CNN Avanzadas: ResNet, Inception y EfficientNet</h1>
<br/><br/>
Dr. Nicolás Gálvez Ramírez<br/>
Dr. Patricio Olivares Roncagliolo<br/><br/>
Ingeniería Civil Telemática<br/>
Departamento de Electrónica<br/>
Universidad Técnica Federico Santa María
</div>
<br>
Fuentes:
<br>
"Hands-on Machine Learning with Scikit-Learn, Keras & TensorFlow"


## 🎯 Objetivos de Aprendizaje

Al finalizar esta clase, el estudiante será capaz de:

1. **Comprender** las innovaciones clave de VGG, ResNet, Inception, MobileNet y EfficientNet.
2. **Explicar** las conexiones residuales (*skip connections*) y su papel frente al *vanishing gradient*.
3. **Aplicar** *transfer learning* con modelos preentrenados en ImageNet (Keras Applications), distinguiendo *feature extraction* de *fine-tuning*.
4. **Implementar** los bloques fundamentales (residual, Inception y MBConv) con la API funcional de Keras.
5. **Comparar** los *trade-offs* entre *accuracy*, número de parámetros y latencia de cada arquitectura.

> 💡 **Prerrequisitos:** [04_RedesNeuronales](../../04_RedesNeuronales/04_RedesNeuronales.ipynb) — perceptrón, *backpropagation* y CNN básicas (`Conv2D`, *pooling*).

## 1. Redes Neuronales, CNNs y Motivación para Redes Más Profundas

Las **redes neuronales (NN)** aprenden representaciones jerárquicas que permiten resolver tareas como clasificación de imágenes y reconocimiento de patrones. Las **redes neuronales convolucionales (CNN)** son NN especializadas para datos con estructura espacial —como imágenes— que comparten pesos mediante filtros de convolución y así detectan patrones locales (bordes, texturas, formas) de forma eficiente.

Para problemas simples basta con una CNN pequeña, pero a medida que la tarea se vuelve más compleja necesitamos **mayor capacidad de representación**. Una forma directa de conseguirla es **agregar más capas**, de modo que la red aprenda características cada vez más abstractas (bordes → texturas → partes → objetos).

Esta necesidad de mayor profundidad guió la **evolución histórica de las CNN**:

$$\text{AlexNet (2012)} \;\rightarrow\; \text{VGG (2014)} \;\rightarrow\; \text{Inception (2014)} \;\rightarrow\; \text{ResNet (2015)} \;\rightarrow\; \text{EfficientNet (2019)}$$

Cada arquitectura resolvió una limitación de la anterior: VGG mostró el poder de apilar filtros pequeños ($3\times3$); Inception introdujo el procesamiento multiescala; ResNet hizo entrenable la gran profundidad con conexiones residuales; y EfficientNet optimizó el balance entre precisión y costo. A lo largo del notebook recorreremos estas ideas.

### Ejemplo: comparación de dos redes CNN con el dataset CIFAR-10

Comparamos una CNN sencilla contra una más profunda, entrenadas sobre **CIFAR-10** (60 000 imágenes a color de $32\times32$ px en 10 clases), para observar el efecto de la profundidad.

In [ ]:
# Cargar CIFAR-10 y definir dos CNN (simple y profunda) para comparar el efecto de la profundidad
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

from tensorflow.keras.datasets import cifar10
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping


# Cargar datos CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
y_train, y_test = to_categorical(y_train, 10), to_categorical(y_test, 10)


# Modelo sencillo
simple_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax')
])
simple_model.compile(optimizer='adam',
                     loss='categorical_crossentropy',
                     metrics=['accuracy'])

# Modelo más profundo
deep_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    Conv2D(32, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Conv2D(64, (3,3), activation='relu'),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(10, activation='softmax')
])
deep_model.compile(optimizer='adam',
                   loss='categorical_crossentropy',
                   metrics=['accuracy'])

# Definir EarlyStopping
early_stop = EarlyStopping(monitor='val_loss',
                           patience=5, 
                           restore_best_weights=True)


In [ ]:
# Entrenamiento corto para comparar
simple_history = simple_model.fit(x_train, y_train, epochs=100, batch_size=64, validation_data=(x_test, y_test), callbacks=[early_stop])

In [ ]:
# Entrenar la CNN mas profunda
deep_history = deep_model.fit(x_train, y_train, epochs=100, batch_size=64, validation_data=(x_test, y_test), callbacks=[early_stop])

In [ ]:
# Comparar la precision de validacion de ambos modelos
print(f"Precisión modelo simple: {simple_history.history['val_accuracy'][-1]:.4f}")
print(f"Precisión modelo más profundo: {deep_history.history['val_accuracy'][-1]:.4f}")

- Este ejemplo muestra que una red más profunda puede lograr mejores resultados en el mismo problema.
- Sin embargo, aumentar la profundidad también trae **nuevos retos**, que veremos a continuación.

## 2. El Desafío de la Profundidad

Aumentar la cantidad de capas **no siempre** mejora el rendimiento:

- Redes demasiado profundas pueden tener **peor precisión** que redes más pequeñas.
- Este fenómeno se conoce como **problema de degradación** (*degradation problem*): al añadir capas, el error de entrenamiento deja de bajar e incluso aumenta.
- No es lo mismo que el **sobreajuste** (*overfitting*): aquí la red profunda tiene mayor error incluso en el conjunto de **entrenamiento**, no solo en validación.
- Al retropropagar el error aparecen dificultades numéricas como la **desaparición** o **explosión** de gradientes.

Estos problemas motivaron nuevas arquitecturas capaces de entrenar redes profundas de forma estable.

In [ ]:
# CNN 'sobrecompleja' (muchas capas sin skip connections): ilustra el problema de degradacion
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping

# Red CNN sobrecompleja para CIFAR-10
overdeep_model = Sequential()
overdeep_model.add(Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)))

# Añadir muchas capas sin skip connections
for _ in range(10):
    overdeep_model.add(Conv2D(32, (3,3), activation='relu', padding='same'))

overdeep_model.add(MaxPooling2D(2,2))
overdeep_model.add(Flatten())
overdeep_model.add(Dense(64, activation='relu'))
overdeep_model.add(Dense(10, activation='softmax'))

overdeep_model.compile(optimizer='adam',
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

# Definir EarlyStopping
early_stop = EarlyStopping(monitor='val_loss',
                           patience=5,  
                           restore_best_weights=True)


# Entrenar misma cantidad de épocas para comparar
overdeep_history = overdeep_model.fit(x_train, y_train,
                                      epochs=100,
                                      batch_size=64,
                                      validation_data=(x_test, y_test),
                                      callbacks=[early_stop])

print(f"Precisión modelo sobrecomplejo: {overdeep_history.history['val_accuracy'][-1]:.4f}")


In [ ]:
# Comparar la precision de los tres modelos (simple, profundo y sobrecomplejo)
print(f"Precisión modelo simple: {simple_history.history['val_accuracy'][-1]:.4f}")
print(f"Precisión modelo más profundo: {deep_history.history['val_accuracy'][-1]:.4f}")
print(f"Precisión modelo sobrecomplejo: {overdeep_history.history['val_accuracy'][-1]:.4f}")

- Esta red, mucho más profunda, puede terminar rindiendo **peor** que la red más pequeña.
- Un rendimiento más bajo incluso en entrenamiento evidencia el **problema de degradación** (no es sobreajuste).
- Para resolverlo se necesitan nuevas **formas de arquitectura**, como las conexiones residuales.

### Vanishing y Exploding Gradients

En redes profundas, el gradiente de la función de pérdida se propaga hacia atrás multiplicando las derivadas de cada capa (regla de la cadena). Para una red de $L$ capas, el gradiente respecto a los pesos de una capa temprana incluye un producto de la forma:

$$\frac{\partial \mathcal{L}}{\partial W_l} \;\propto\; \prod_{k=l+1}^{L} \frac{\partial a_k}{\partial a_{k-1}}$$

- Si los factores son **menores que 1**, el producto tiende a **0**: los gradientes **se desvanecen** (*vanishing*) y las capas iniciales casi no aprenden.
- Si los factores son **mayores que 1**, el producto **explota** (*exploding*) y el entrenamiento se vuelve inestable (pérdida `NaN`).
- La **normalización de lotes** (*Batch Normalization*) y una buena inicialización ayudan, pero no siempre bastan.

En la siguiente sección veremos cómo las **conexiones residuales** ofrecen un camino directo para el gradiente.

## 3. ResNet y Conexiones Residuales

Para resolver los problemas de degradación y de gradientes, He *et al.* (2015) propusieron **ResNet** (*Residual Network*), introduciendo las **conexiones residuales** o *skip connections*.

En lugar de pedirle a un bloque de capas que aprenda una transformación completa $H(\mathbf{x})$, se le pide que aprenda solo el **residuo** $\mathcal{F}(\mathbf{x}) = H(\mathbf{x}) - \mathbf{x}$, y la salida se reconstruye sumando la entrada original:

$$\mathbf{y} = \mathcal{F}(\mathbf{x}, \{W_i\}) + \mathbf{x}$$

- Si la transformación óptima es la **identidad**, basta con llevar $\mathcal{F}(\mathbf{x}) \to 0$, algo mucho más fácil de aprender.
- La suma directa crea una "autopista" para el gradiente, **mitigando el *vanishing gradient***.
- En la práctica, ResNet permite entrenar redes de **cientos de capas** sin perder rendimiento.

### Bloques residuales

<img src="images/resnet.png" alt="Diagrama de un bloque residual con skip connection" width="400px"/>

El **bloque residual** combina la entrada original con la salida de un par de capas convolucionales:

- Dos `Conv2D` + `BatchNormalization` → transforman la entrada (calculan el residuo $\mathcal{F}(\mathbf{x})$).
- `Add()` → suma la entrada original (**skip connection**), facilitando aprender la identidad.
- `ReLU` final → mantiene la no linealidad.

**Justificación:**
- Ayuda a entrenar redes profundas sin degradación.
- Mejora el flujo de gradientes.
- Permite aprender ajustes finos sobre la entrada sin perder información.

> Cuando cambian las dimensiones (por *stride* o número de filtros), la conexión de atajo usa una **proyección** `Conv2D` de $1\times1$ para igualar las formas antes de sumar.

In [ ]:
# Función que define un bloque residual básico
def residual_block(x, filters, stride=1, use_projection=False):
    shortcut = x

    # Ruta principal
    y = layers.Conv2D(filters, 3, strides=stride, padding='same')(x)
    y = layers.BatchNormalization()(y)
    y = layers.ReLU()(y)
    y = layers.Conv2D(filters, 3, padding='same')(y)
    y = layers.BatchNormalization()(y)

    # Proyección si cambia tamaño o filtros
    if use_projection:
        shortcut = layers.Conv2D(filters, 1, strides=stride, padding='same')(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    out = layers.Add()([shortcut, y])
    out = layers.ReLU()(out)
    return out

def build_simple_resnet(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    # Stem
    x = layers.Conv2D(32, 3, strides=1, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    # Etapa 1
    x = residual_block(x, filters=32)
    x = residual_block(x, filters=32)

    # Etapa 2 con downsampling
    x = residual_block(x, filters=64, stride=2, use_projection=True)
    x = residual_block(x, filters=64)

    # Etapa 3 con downsampling
    x = residual_block(x, filters=128, stride=2, use_projection=True)
    x = residual_block(x, filters=128)

    # Clasificador
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inputs, outputs, name="SimpleResNet")
    return model


In [ ]:
# Crear y compilar SimpleResNet
resnet_model = build_simple_resnet((32, 32, 3), 10)
resnet_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
# Configurar EarlyStopping y entrenar SimpleResNet
from tensorflow.keras.callbacks import EarlyStopping

# Callback EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Hiperparámetros
batch_size = 64
epochs = 100

# Entrenar SimpleResNet
history_resnet = resnet_model.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(x_test, y_test),
    callbacks=[early_stop]
)

In [ ]:
# Mostrar la precision de validacion de SimpleResNet
print(f"Precisión modelo SimpleResNet: {history_resnet.history['val_accuracy'][-1]:.4f}")

### ResNet50 entrenada desde cero con CIFAR-10

- Keras incluye arquitecturas como **ResNet50** listas para usar (`tensorflow.keras.applications`).
- Aquí se carga ResNet50 **sin pesos preentrenados** (`weights=None`) para entrenarla desde cero.
- Se reemplaza la capa final (*top*) por una capa densa que clasifica las 10 clases de CIFAR-10.
- Esto permite comparar su rendimiento con las redes planas y con `SimpleResNet`.

In [ ]:
# Cargar ResNet50 (sin pesos) con un pipeline tf.data y entrenarla desde cero en CIFAR-10
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

# Cargar CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
num_classes = 10

# Normalizar y codificar etiquetas
y_train = to_categorical(y_train, num_classes)
y_test = to_categorical(y_test, num_classes)

# Crear datasets tf.data con redimensionamiento
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, (224, 224))
    return image, label

batch_size = 64

# Crea los conjuntos de datos de entrenamiento y validación usando la API tf.data:
# - from_tensor_slices: convierte los tensores (x, y) en un dataset
# - shuffle: mezcla los datos (solo en entrenamiento)
# - map: aplica la función de preprocesamiento en paralelo
# - batch: agrupa los datos en lotes del tamaño especificado
# - prefetch: prepara el siguiente lote mientras se entrena para mejorar el rendimiento

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(buffer_size=5000)
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

# Cargar ResNet50 sin pesos preentrenados
base_model = ResNet50(weights=None, include_top=False, input_shape=(224, 224, 3))

# Construir modelo final
inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)

# Compilar
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Entrenar usando dataset eficiente
history_resnet50 = model.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds,
    callbacks=[early_stop]
)

# Precisión final
print(f"Precisión ResNet50 desde cero: {history_resnet50.history['val_accuracy'][-1]:.4f}")


In [ ]:
# Mostrar la precision de validacion de ResNet50
print(f"Precisión modelo ResNet50: {history_resnet50.history['val_accuracy'][-1]:.4f}")

### ¿Qué demuestra ResNet?

- Apilar más capas no siempre mejora el rendimiento: puede aparecer el **problema de degradación**.
- Las **conexiones residuales** permiten que la red aprenda la función identidad fácilmente y ajuste solo lo necesario.
- Esto mantiene el flujo de gradientes y facilita entrenar redes profundas de forma estable.
- ResNet abrió la puerta a arquitecturas con **cientos de capas**, hoy omnipresentes en visión por computador.
- En la práctica se logra mejor precisión y convergencia más estable que con redes planas de profundidad comparable.

## 4. Inception (GoogLeNet)

**Inception** es una arquitectura profunda diseñada para procesar imágenes de forma más **eficiente** en cómputo. Su idea central es el **módulo Inception**: en lugar de elegir un único tamaño de filtro, aplica **varios en paralelo** ($1\times1$, $3\times3$, $5\times5$) más *pooling*, y concatena sus salidas.

- Cada rama captura patrones a una **escala distinta**; la concatenación combina información multiescala.
- Las convoluciones $1\times1$ actúan como **cuellos de botella** que reducen el número de canales antes de las convoluciones costosas ($3\times3$, $5\times5$), bajando el costo computacional.
- Fue popularizada por **GoogLeNet** (Inception v1, 2014) y mejorada en las versiones v2, v3 y v4.
- Destaca por lograr alta precisión con relativamente **pocos parámetros** (~7M en v1, frente a ~138M de VGG-16).

### El bloque Inception

<img src="images/inception.png" alt="Diagrama de un módulo Inception con ramas en paralelo" width="600px"/>

Un bloque Inception combina **varias ramas de filtros en paralelo**.

> **¿Qué es una rama?** Un camino independiente que aplica una secuencia de convoluciones o *pooling* sobre la **misma entrada**. Cada rama extrae características diferentes y, al final, todas se **concatenan** (a lo largo del eje de canales) para formar la salida del bloque.

**Ramas típicas en Inception v1:**
- **Rama 1:** conv $1\times1$ — características básicas.
- **Rama 2:** conv $1\times1$ → conv $3\times3$ — detalles locales.
- **Rama 3:** conv $1\times1$ → conv $5\times5$ — patrones más amplios.
- **Rama 4:** *max pooling* $3\times3$ → conv $1\times1$ — robustez y reducción de dimensiones.

Esta combinación captura información a múltiples escalas dentro de un solo bloque manteniendo la eficiencia. Por eso los bloques Inception se **apilan** para construir arquitecturas profundas como GoogLeNet.

In [ ]:
# Definir un modulo Inception (ramas en paralelo) y construir un modelo de ejemplo
from tensorflow.keras import layers, Model, Input

def inception_block(x, f1x1, f3x3_reduce, f3x3, f5x5_reduce, f5x5, pool_proj):
    # Rama 1: 1x1 conv
    path1 = layers.Conv2D(f1x1, (1, 1), padding='same', activation='relu')(x)
    
    # Rama 2: 1x1 conv + 3x3 conv
    path2 = layers.Conv2D(f3x3_reduce, (1, 1), padding='same', activation='relu')(x)
    path2 = layers.Conv2D(f3x3, (3, 3), padding='same', activation='relu')(path2)
    
    # Rama 3: 1x1 conv + 5x5 conv
    path3 = layers.Conv2D(f5x5_reduce, (1, 1), padding='same', activation='relu')(x)
    path3 = layers.Conv2D(f5x5, (5, 5), padding='same', activation='relu')(path3)
    
    # Rama 4: 3x3 max pooling + 1x1 conv
    path4 = layers.MaxPooling2D((3, 3), strides=(1, 1), padding='same')(x)
    path4 = layers.Conv2D(pool_proj, (1, 1), padding='same', activation='relu')(path4)
    
    # Concatenar todas las ramas
    output = layers.concatenate([path1, path2, path3, path4], axis=-1)
    return output

# Ejemplo de uso del bloque
inputs = Input(shape=(32, 32, 192))
x = inception_block(inputs, 64, 96, 128, 16, 32, 32)
model = Model(inputs, x)
model.summary()

### Prueba de InceptionV3

- La arquitectura **Inception** usa bloques que combinan filtros de distintos tamaños para capturar patrones a varias escalas.
- **InceptionV3** es una versión optimizada y muy popular para clasificación de imágenes; espera entradas de $299\times299$ px.
- Aquí se construye InceptionV3 (`weights=None`) y se adapta su capa final a las 10 clases de CIFAR-10, redimensionando las imágenes a la resolución esperada.
- El ejemplo permite entender cómo se usan modelos grandes con la API de Keras.

In [ ]:
# Cargar InceptionV3 (sin pesos) con un pipeline tf.data y entrenarla desde cero en CIFAR-10
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

# Cargar CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
num_classes = 10

# Crear tf.data.Dataset para redimensionar
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, (299, 299))
    return image, tf.one_hot(label[0], num_classes)

batch_size = 64

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(5000)
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

# Cargar InceptionV3 sin pesos y sin top layer
base_model = InceptionV3(weights=None, include_top=False, input_shape=(299, 299, 3))

# Agregar capa final para CIFAR-10
inputs = tf.keras.Input(shape=(299, 299, 3))
x = base_model(inputs)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)

# Compilar
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Entrenar
history_inception = model.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds,
    callbacks=[early_stop]
)

# Mostrar accuracy final
print(f"Precisión InceptionV3 desde cero: {history_inception.history['val_accuracy'][-1]:.4f}")



In [ ]:
# Mostrar accuracy final
print(f"Precisión InceptionV3 desde cero: {history_inception.history['val_accuracy'][-1]:.4f}")

### Resumen de Inception

- Inception muestra cómo una arquitectura bien diseñada puede capturar patrones a **múltiples escalas** dentro de un mismo módulo.
- Aunque es más compleja que una CNN tradicional, su uso eficiente de filtros paralelos y de convoluciones $1\times1$ la hace **competitiva y económica** en parámetros.

## 5. EfficientNet

**EfficientNet** (Tan & Le, 2019) es una familia de CNN optimizadas para obtener la **mejor relación entre precisión y eficiencia computacional**.

Su idea central es el **compound scaling**: escalar de forma **balanceada y sistemática** la profundidad ($d$), la anchura ($w$, número de filtros) y la resolución de entrada ($r$) mediante un único coeficiente $\phi$:

$$d = \alpha^{\phi}, \quad w = \beta^{\phi}, \quad r = \gamma^{\phi} \qquad \text{con} \qquad \alpha \cdot \beta^{2} \cdot \gamma^{2} \approx 2$$

- Los coeficientes base $\alpha, \beta, \gamma$ se obtienen por búsqueda de arquitecturas (**AutoML** / NAS).
- La unidad básica es el bloque **MBConv**, heredado de **MobileNetV2**, que combina:
  - Convolución de expansión ($1\times1$).
  - **Depthwise Separable Convolution** (mucho más eficiente).
  - Bloque **Squeeze-and-Excitation (SE)** para recalibrar la importancia de cada canal.
- Gracias a esta estructura, EfficientNet logra alta precisión con **menos parámetros** que ResNet o Inception (EfficientNet-B0 ≈ 5.3M).

### El bloque MBConv

El **MBConv (Mobile Inverted Bottleneck Convolution)** es el bloque principal de EfficientNet. Combina varias ideas clave:

- **Expansión:** aumenta temporalmente el número de canales con una convolución $1\times1$.
- **Depthwise Convolution:** aplica un filtro independiente por canal, reduciendo drásticamente los cálculos.
- **Proyección:** reduce de nuevo los canales con otra convolución $1\times1$.
- **Squeeze-and-Excitation (SE):** recalibra los canales para resaltar los más relevantes.

El término *Inverted Bottleneck* indica que, a diferencia de un cuello de botella clásico (reducir → procesar → expandir), aquí se hace **expandir → procesar → reducir**.

**Depthwise Separable Convolution.** Es la clave de eficiencia de **MobileNet** y EfficientNet: descompone una convolución estándar en dos pasos —una convolución *depthwise* (un filtro por canal) seguida de una convolución *pointwise* ($1\times1$, que mezcla canales)—. Para $N$ filtros de salida y kernels de $D_K\times D_K$, el costo se reduce aproximadamente en un factor:

$$\frac{1}{N} + \frac{1}{D_K^{2}}$$

Es decir, con kernels $3\times3$ el costo baja alrededor de **8–9×** respecto a una convolución estándar, logrando **más precisión con menor costo computacional**.

In [ ]:
# Definir los bloques Squeeze-and-Excitation y MBConv (base de EfficientNet) y probar uno
from tensorflow.keras import layers, Model, Input

def squeeze_excite_block(inputs, ratio=4):
    # Bloque SE: recalibra la importancia de cada canal de características
    filters = inputs.shape[-1]
    se = layers.GlobalAveragePooling2D()(inputs)
    se = layers.Dense(filters // ratio, activation='relu')(se)
    se = layers.Dense(filters, activation='sigmoid')(se)
    se = layers.Reshape((1, 1, filters))(se)
    return layers.multiply([inputs, se])

def mbconv_block(x, expansion_factor=6, filters=16, stride=1):
    # Bloque MBConv: base de EfficientNet (expandir → procesar → reducir)
    in_channels = x.shape[-1]
    # Expandir
    x_expanded = layers.Conv2D(in_channels * expansion_factor, 1, padding='same', activation='relu')(x)
    x_expanded = layers.BatchNormalization()(x_expanded)

    # Depthwise separable convolution
    x_depthwise = layers.DepthwiseConv2D(3, strides=stride, padding='same', activation='relu')(x_expanded)
    x_depthwise = layers.BatchNormalization()(x_depthwise)

    # Squeeze-and-Excitation
    x_se = squeeze_excite_block(x_depthwise)

    # Proyección
    x_projected = layers.Conv2D(filters, 1, padding='same')(x_se)
    x_projected = layers.BatchNormalization()(x_projected)

    # Skip connection si no hay cambio de forma
    if stride == 1 and in_channels == filters:
        x_out = layers.Add()([x, x_projected])
    else:
        x_out = x_projected

    return x_out

# Ejemplo de uso: entrada y un bloque MBConv
inputs = Input(shape=(32, 32, 16))
x = mbconv_block(inputs, expansion_factor=6, filters=16, stride=1)
model = Model(inputs, x)
model.summary()


### Prueba de EfficientNetB0

- **EfficientNetB0** es la versión base de la familia, diseñada para un buen equilibrio entre precisión y eficiencia.
- Keras permite cargar EfficientNetB0 con pesos preentrenados en ImageNet (`weights='imagenet'`) o desde cero (`weights=None`).

> **Transfer learning.** Reutilizar un modelo entrenado en un gran dataset (ImageNet) para una tarea nueva. Existen dos modalidades:
> - **Feature extraction:** se **congela** la red base (`base_model.trainable = False`) y solo se entrena el clasificador final. Rápido y útil con pocos datos.
> - **Fine-tuning:** se **descongelan** algunas (o todas) las capas superiores del *backbone* y se reentrenan con una tasa de aprendizaje pequeña para adaptarlas al nuevo dominio.
>
> En el ejemplo siguiente se entrena desde cero (`weights=None`); para *transfer learning*, basta cambiar a `weights='imagenet'` y decidir qué capas congelar.

In [ ]:
# Cargar EfficientNetB0 (sin pesos) con un pipeline tf.data y entrenarla en CIFAR-10
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

# Cargar CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
num_classes = 10

# Dataset tf.data con resize dinámico
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = tf.image.resize(image, (224, 224))
    return image, tf.one_hot(label[0], num_classes)

batch_size = 64

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(5000)
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

# Cargar EfficientNetB0 SIN pesos preentrenados (o pon weights='imagenet' para transfer learning)
base_model = EfficientNetB0(weights=None, include_top=False, input_shape=(224, 224, 3))

# Construir modelo final para CIFAR-10
inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs)
x = layers.GlobalAveragePooling2D()(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)

# Compilar
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Entrenar
history_efficientnet = model.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds,
    callbacks=[early_stop]
)

In [ ]:
# Mostrar precisión final en validación
print(f"Precisión EfficientNetB0 desde cero: {history_efficientnet.history['val_accuracy'][-1]:.4f}")

## 6. Técnicas de Optimización y Regularización en Redes Profundas

En esta sección revisamos estrategias prácticas para mejorar el **rendimiento** y la **generalización** de las CNN. Estas técnicas permiten:

- Acelerar y estabilizar el entrenamiento.
- Hacer los modelos más robustos frente a nuevas muestras.
- Reducir el sobreajuste.

Veremos de forma resumida:
- **Batch Normalization** — estabiliza las activaciones internas.
- **Data Augmentation avanzada** — con **Cutout** y **Mixup**.
- **Label Smoothing** — suaviza las etiquetas para evitar la sobreconfianza.

### Batch Normalization

**Batch Normalization (BN)** normaliza las activaciones de cada mini-lote para estabilizar el entrenamiento. Para un mini-lote $\mathcal{B}$ con media $\mu_\mathcal{B}$ y varianza $\sigma_\mathcal{B}^{2}$:

$$\hat{x} = \frac{x - \mu_\mathcal{B}}{\sqrt{\sigma_\mathcal{B}^{2} + \epsilon}}, \qquad y = \gamma\,\hat{x} + \beta$$

- Reduce el **Internal Covariate Shift** (el cambio de distribución de las activaciones internas durante el entrenamiento).
- Incluye parámetros **aprendibles** $\gamma$ (escala) y $\beta$ (desplazamiento) que preservan la capacidad expresiva.
- Beneficios: entrenamiento más rápido y estable, y un ligero efecto **regularizador**.
- Normalmente se coloca después de `Conv2D` / `Dense` y antes de la activación.

In [ ]:
# Comparar el entrenamiento de una CNN con y sin Batch Normalization
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

# Dataset CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

num_classes = 10
y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)

# Modelo SIN Batch Normalization
def build_model_no_bn(input_shape, num_classes):
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=input_shape),
        layers.Conv2D(32, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.Conv2D(64, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Modelo CON Batch Normalization
def build_model_with_bn(input_shape, num_classes):
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.MaxPooling2D((2, 2)),
        
        layers.Flatten(),
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# Crear ambos modelos
model_no_bn = build_model_no_bn(x_train.shape[1:], num_classes)
model_with_bn = build_model_with_bn(x_train.shape[1:], num_classes)

# Compilar ambos
model_no_bn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_with_bn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# EarlyStopping
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Entrenar ambos modelos para comparar
print("\nEntrenando modelo SIN Batch Normalization")
history_no_bn = model_no_bn.fit(
    x_train, y_train_cat,
    batch_size=64,
    epochs=100,
    validation_data=(x_test, y_test_cat),
    callbacks=[early_stop]
)

print("\nEntrenando modelo CON Batch Normalization")
history_with_bn = model_with_bn.fit(
    x_train, y_train_cat,
    batch_size=64,
    epochs=100,
    validation_data=(x_test, y_test_cat),
    callbacks=[early_stop]
)

# Mostrar precisión final
print(f"\nPrecisión sin BN: {history_no_bn.history['val_accuracy'][-1]:.4f}")
print(f"Precisión con BN: {history_with_bn.history['val_accuracy'][-1]:.4f}")

### Data Augmentation

- **Data Augmentation** genera nuevas muestras a partir de los datos reales aplicando transformaciones que preservan la etiqueta.
- Se usa en **imágenes, texto, audio** y otros tipos de datos.
- Aumenta la **diversidad** del dataset sin recolectar más datos.
- Ayuda a que el modelo sea **más robusto** y reduce el sobreajuste.
- Ejemplos clásicos en imágenes: rotaciones, *flips*, desplazamientos y *zoom*.

#### Cutout

- **Cutout** oculta aleatoriamente un **parche cuadrado** dentro de cada imagen (lo pone a cero).
- Obliga al modelo a **no depender de una sola región**, aprendiendo a usar el contexto completo.
- Simula oclusiones reales (por ejemplo, un objeto parcialmente tapado).
- Reduce el riesgo de sobreajuste y mejora la generalización.

In [ ]:
# Visualizar el efecto de Cutout sobre una imagen de CIFAR-10
import numpy as np
import matplotlib.pyplot as plt

def apply_cutout(image, patch_size=16):
    # Aplica Cutout a una sola imagen.
    image_h, image_w, _ = image.shape

    center_x = np.random.randint(0, image_w)
    center_y = np.random.randint(0, image_h)

    x1 = np.clip(center_x - patch_size // 2, 0, image_w)
    x2 = np.clip(center_x + patch_size // 2, 0, image_w)
    y1 = np.clip(center_y - patch_size // 2, 0, image_h)
    y2 = np.clip(center_y + patch_size // 2, 0, image_h)

    cutout_image = image.copy()
    cutout_image[y1:y2, x1:x2, :] = 0
    return cutout_image

# Tomar una imagen de CIFAR-10
sample_image = x_train[0]

plt.figure(figsize=(8, 4))

plt.subplot(1, 2, 1)
plt.imshow(sample_image)
plt.title("Original")
plt.axis('off')

plt.subplot(1, 2, 2)
plt.imshow(apply_cutout(sample_image, patch_size=16))
plt.title("Con Cutout")
plt.axis('off')

plt.show()


In [ ]:
# Entrenar una CNN aplicando Cutout mediante un pipeline tf.data
import tensorflow as tf
import numpy as np

# Cargar CIFAR-10
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
num_classes = 10

# Preprocesar etiquetas
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

# Función Cutout
def cutout(image, patch_size=16):
    h, w, c = image.shape
    y = tf.random.uniform([], 0, h, dtype=tf.int32)
    x = tf.random.uniform([], 0, w, dtype=tf.int32)

    y1 = tf.clip_by_value(y - patch_size // 2, 0, h)
    y2 = tf.clip_by_value(y + patch_size // 2, 0, h)
    x1 = tf.clip_by_value(x - patch_size // 2, 0, w)
    x2 = tf.clip_by_value(x + patch_size // 2, 0, w)

    mask = tf.ones((y2 - y1, x2 - x1, c), dtype=image.dtype)
    mask = tf.pad(mask, [[y1, h - y2], [x1, w - x2], [0, 0]], constant_values=0)
    return image * (1 - mask)

# Pipeline tf.data.Dataset
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    image = cutout(image, patch_size=16)
    return image, label

batch_size = 64

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train_cat))
    .shuffle(5000)
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test_cat))
    .map(lambda x, y: (tf.cast(x, tf.float32) / 255.0, y))
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

# Modelo simple
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation='relu', input_shape=(32, 32, 3)),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# EarlyStopping
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Entrenar
history = model.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds,
    callbacks=[early_stop]
)

print(f"Precisión final con Cutout: {history.history['val_accuracy'][-1]:.4f}")

#### Mixup

**Mixup** combina linealmente dos imágenes y sus etiquetas para crear ejemplos "intermedios":

$$\tilde{x} = \lambda\,x_i + (1-\lambda)\,x_j, \qquad \tilde{y} = \lambda\,y_i + (1-\lambda)\,y_j, \qquad \lambda \sim \text{Beta}(\alpha, \alpha)$$

- Suaviza las fronteras de decisión y **evita la sobreconfianza** → mejor generalización y robustez.
- Crea ejemplos "intermedios" entre clases, muy útil para datasets pequeños o ruidosos.

In [ ]:
# Visualizar el efecto de Mixup combinando dos imagenes de CIFAR-10
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

def apply_mixup(image1, image2, lam=0.5):
    # Combina dos imágenes usando Mixup.
    return lam * image1 + (1 - lam) * image2


(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
num_classes = 10

# Tomar dos imágenes de CIFAR-10
img1 = x_train[0]
img2 = x_train[1]

# Coeficiente de mezcla (puedes probar otros)
lam = 0.6

# Aplicar Mixup
mixed_image = apply_mixup(img1, img2, lam)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(img1)
plt.title("Imagen 1")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(img2)
plt.title("Imagen 2")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(mixed_image.astype(np.uint8))
plt.title(f"Mixup (λ = {lam:.1f})")
plt.axis('off')

plt.show()


In [ ]:
# Entrenar una CNN aplicando Mixup por lote
import tensorflow as tf
import numpy as np

# Cargar CIFAR-10
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
num_classes = 10

# Preprocesar etiquetas
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

# Función Mixup
def mixup(batch_x, batch_y, alpha=0.2):
    """Aplica Mixup a un batch."""
    batch_size = tf.shape(batch_x)[0]
    lam = np.random.beta(alpha, alpha)
    index = tf.random.shuffle(tf.range(batch_size))

    mixed_x = lam * batch_x + (1 - lam) * tf.gather(batch_x, index)
    mixed_y = lam * batch_y + (1 - lam) * tf.gather(batch_y, index)
    return mixed_x, mixed_y

# Pipeline tf.data.Dataset
def preprocess(image, label):
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

batch_size = 64

# Crear dataset base
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train_cat))
    .shuffle(5000)
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
)

# Aplicar Mixup en cada batch
def mixup_map(batch_x, batch_y):
    return mixup(batch_x, batch_y)

train_ds = train_ds.map(mixup_map, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

# Validación sin Mixup
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test_cat))
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

# Modelo simple
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation='relu', input_shape=(32, 32, 3)),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# EarlyStopping
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Entrenar
history = model.fit(
    train_ds,
    epochs=100,
    validation_data=val_ds,
    callbacks=[early_stop]
)

print(f"Precisión final con Mixup: {history.history['val_accuracy'][-1]:.4f}")

### Label Smoothing

**Label Smoothing** modifica las etiquetas "duras" (*one-hot*) para que no sean 100 % seguras. Con $K$ clases y un factor $\varepsilon$, la etiqueta suavizada es (convención usada por `CategoricalCrossentropy(label_smoothing=ε)`):

$$y_k^{\text{LS}} = (1-\varepsilon)\,y_k + \frac{\varepsilon}{K}$$

- Ejemplo con 4 clases y $\varepsilon = 0.1$: la etiqueta `[0, 1, 0, 0]` se convierte en `[0.025, 0.925, 0.025, 0.025]` (la clase correcta recibe $1-\varepsilon+\varepsilon/K$ y las demás $\varepsilon/K$).
- Efectos sobre el modelo:
  - Lo hace **menos confiado**, evitando el sobreajuste.
  - Aprende fronteras de decisión más suaves.
  - Mejora su **calibración** en casos ambiguos.

In [ ]:
# Entrenar una CNN usando Label Smoothing en la funcion de perdida
import tensorflow as tf

# Cargar CIFAR-10
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
num_classes = 10

# Normalizar y etiquetas
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_cat = tf.keras.utils.to_categorical(y_test, num_classes)

# Modelo simple
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, 3, activation='relu', input_shape=(32, 32, 3)),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

# Compilar con Label Smoothing
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

# EarlyStopping
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Entrenar
history = model.fit(
    x_train, y_train_cat,
    epochs=100,
    batch_size=64,
    validation_data=(x_test, y_test_cat),
    callbacks=[early_stop]
)

print(f"Precisión final con Label Smoothing: {history.history['val_accuracy'][-1]:.4f}")


- Con estas técnicas (**Batch Normalization**, **Data Augmentation** avanzada —Cutout y Mixup— y **Label Smoothing**) podemos entrenar redes profundas de forma más estable y robusta.
- Cada una resuelve un problema distinto: BatchNorm estabiliza las activaciones internas; Cutout y Mixup hacen al modelo más resistente y reducen el sobreajuste; Label Smoothing controla la sobreconfianza y mejora la calibración.
- Todas se combinan fácilmente con arquitecturas avanzadas como ResNet, Inception o EfficientNet.
- Probar y ajustar estas estrategias es clave para lograr modelos confiables y con mejor capacidad de generalización en escenarios reales.

## 📌 Resumen

Recorrimos la evolución de las arquitecturas CNN y las técnicas que las hacen entrenables y eficientes.

| Arquitectura | Año | Innovación clave | Profundidad | Parámetros (aprox.) |
|--------------|-----|------------------|-------------|---------------------|
| **AlexNet** | 2012 | ReLU, Dropout, entrenamiento en GPU | 8 | ~60M |
| **VGG-16** | 2014 | Apilar filtros pequeños $3\times3$ | 16 | ~138M |
| **Inception v1 (GoogLeNet)** | 2014 | Módulos multiescala + conv $1\times1$ | 22 | ~7M |
| **ResNet-50** | 2015 | Conexiones residuales (*skip connections*) | 50 | ~25M |
| **MobileNet** | 2017 | *Depthwise separable convolutions* | 28 | ~4M |
| **EfficientNet-B0** | 2019 | *Compound scaling* + MBConv + SE | — | ~5.3M |

**Ideas transversales:**

| Concepto | Descripción breve |
|----------|-------------------|
| *Vanishing gradient* | Los gradientes se desvanecen en redes muy profundas; las conexiones residuales lo mitigan. |
| Conexión residual | $\mathbf{y} = \mathcal{F}(\mathbf{x}) + \mathbf{x}$; facilita aprender la identidad y el flujo del gradiente. |
| Convolución $1\times1$ | Reduce/mezcla canales de forma barata; clave en Inception y MBConv. |
| *Depthwise separable conv* | Factoriza la convolución (*depthwise* + *pointwise*); ~8–9× menos costo. |
| *Transfer learning* | Reutilizar pesos de ImageNet: *feature extraction* (congelar) vs. *fine-tuning* (reentrenar). |
| *Trade-off* | Más precisión suele implicar más parámetros y mayor latencia; EfficientNet optimiza el balance. |

### 🔗 Próximos pasos
- Aplicar **transfer learning** real cargando `weights='imagenet'` y comparando *feature extraction* vs. *fine-tuning*.
- Explorar modelos generativos en [05-3 GAN y Autoencoders](../05-3_GAN_AE/) y atención en [05-2 Transformers](../05-2_Transformers/).
- Lecturas recomendadas: ver la sección de Referencias.

## 📚 Referencias

**Arquitecturas**
- **[Krizhevsky et al., 2012]** ImageNet Classification with Deep Convolutional Neural Networks (*AlexNet*). *NeurIPS*.
- **[Simonyan & Zisserman, 2014]** Very Deep Convolutional Networks for Large-Scale Image Recognition (*VGG*). [arXiv:1409.1556](https://arxiv.org/abs/1409.1556)
- **[Szegedy et al., 2015]** Going Deeper with Convolutions (*GoogLeNet / Inception*). *CVPR*. [arXiv:1409.4842](https://arxiv.org/abs/1409.4842)
- **[He et al., 2016]** Deep Residual Learning for Image Recognition (*ResNet*). *CVPR*. [arXiv:1512.03385](https://arxiv.org/abs/1512.03385)
- **[Howard et al., 2017]** MobileNets: Efficient CNNs for Mobile Vision Applications. [arXiv:1704.04861](https://arxiv.org/abs/1704.04861)
- **[Tan & Le, 2019]** EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks. *ICML*. [arXiv:1905.11946](https://arxiv.org/abs/1905.11946)

**Técnicas de entrenamiento**
- **[Ioffe & Szegedy, 2015]** Batch Normalization: Accelerating Deep Network Training. [arXiv:1502.03167](https://arxiv.org/abs/1502.03167)
- **[DeVries & Taylor, 2017]** Improved Regularization of CNNs with Cutout. [arXiv:1708.04552](https://arxiv.org/abs/1708.04552)
- **[Zhang et al., 2018]** mixup: Beyond Empirical Risk Minimization. *ICLR*. [arXiv:1710.09412](https://arxiv.org/abs/1710.09412)

**Documentación y libros**
- **[Keras Applications]** Modelos preentrenados: https://keras.io/api/applications/
- **[TensorFlow]** Transfer learning y fine-tuning: https://www.tensorflow.org/tutorials/images/transfer_learning
- **[Géron, 2022]** *Hands-on Machine Learning with Scikit-Learn, Keras & TensorFlow*, 3.ª ed. O'Reilly.